# Notebook 03 — RAG 파이프라인
## TechDocRAG: 멀티모달 기술 문서 RAG
### 이 노트북에서 배울 것
- 하이브리드 검색 클래스 구현 (Dense + BM25 앙상블)
- Qwen2.5-VL (Ollama) 에 텍스트 + 페이지 이미지 동시 주입
- Rule-based 폴백 (Ollama 미설치 시 자동 전환)
- 검색 정확도 평가 (Hit Rate @ K, MRR)

In [ ]:
# 한글 폰트 + 기본 설정
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import os, sys, json, time, re, pickle, base64, urllib.request
from pathlib import Path
from dataclasses import dataclass

ROOT       = Path().absolute().parent
OUTPUT_DIR = ROOT / 'output'
VECTOR_DIR = ROOT / 'vector_db'

# NB01 청크 + NB02 BM25 인덱스 로드
with open(OUTPUT_DIR / 'chunks.json', 'r', encoding='utf-8') as f:
    chunks = json.load(f)

with open(OUTPUT_DIR / 'bm25_index.pkl', 'rb') as f:
    bm25_data = pickle.load(f)

bm25       = bm25_data['bm25']
chunk_ids  = bm25_data['chunk_ids']

print(f"청크 로드: {len(chunks)}개")
print(f"BM25 인덱스 로드: {len(chunk_ids)}개 문서")

In [ ]:
# BGE-M3 + ChromaDB 로드
from sentence_transformers import SentenceTransformer
import chromadb

embedder   = SentenceTransformer('BAAI/bge-m3', cache_folder=str(ROOT / 'models'))
chroma     = chromadb.PersistentClient(path=str(VECTOR_DIR))
collection = chroma.get_collection("tech_docs")

print(f"BGE-M3 로드 완료 (dim={embedder.get_sentence_embedding_dimension()})")
print(f"ChromaDB 컬렉션 문서 수: {collection.count()}")

## 1. 하이브리드 검색 클래스

In [ ]:
@dataclass
class RetrievedChunk:
    chunk_id:     str
    text:         str
    image_path:   str
    page_num:     int
    doc_name:     str
    content_type: str
    hybrid_score: float
    dense_score:  float
    bm25_score:   float


class HybridRetriever:
    """Dense (BGE-M3) + BM25 앙상블 검색기"""

    def __init__(self, dense_weight: float = 0.7):
        self.dense_w = dense_weight
        self._chunk_map = {c['chunk_id']: c for c in chunks}

    def _tokenize(self, text: str) -> list[str]:
        return re.findall(r'[a-z0-9]+|[가-힣]{2,}', text.lower())

    def search(self, query: str, top_k: int = 3) -> list[RetrievedChunk]:
        # 1) Dense 검색
        q_emb = embedder.encode([query], normalize_embeddings=True)
        dense_res = collection.query(
            query_embeddings=q_emb.tolist(),
            n_results=min(top_k * 2, collection.count()),
            include=['documents', 'metadatas', 'distances']
        )
        dense_scores = {
            dense_res['ids'][0][i]: 1 - dense_res['distances'][0][i]
            for i in range(len(dense_res['ids'][0]))
        }

        # 2) BM25 검색
        q_tokens  = self._tokenize(query)
        bm25_raw  = bm25.get_scores(q_tokens)
        bm25_max  = max(bm25_raw) if max(bm25_raw) > 0 else 1
        bm25_norm = {chunk_ids[i]: float(bm25_raw[i]) / bm25_max
                     for i in range(len(chunk_ids))}

        # 3) 앙상블 + 정렬
        all_ids = set(dense_scores) | set(k for k, v in bm25_norm.items() if v > 0)
        hybrid  = {
            cid: self.dense_w * dense_scores.get(cid, 0.0)
                 + (1 - self.dense_w) * bm25_norm.get(cid, 0.0)
            for cid in all_ids
        }
        ranked = sorted(hybrid.items(), key=lambda x: x[1], reverse=True)[:top_k]

        results = []
        for cid, h_score in ranked:
            c = self._chunk_map.get(cid)
            if c:
                results.append(RetrievedChunk(
                    chunk_id=cid,
                    text=c['text'],
                    image_path=c['image_path'],
                    page_num=c['page_num'],
                    doc_name=c['doc_name'],
                    content_type=c['content_type'],
                    hybrid_score=round(h_score, 4),
                    dense_score=round(dense_scores.get(cid, 0.0), 4),
                    bm25_score=round(bm25_norm.get(cid, 0.0), 4)
                ))
        return results


retriever = HybridRetriever(dense_weight=0.7)
print("HybridRetriever 초기화 완료 (Dense 70% + BM25 30%)")

In [ ]:
# 검색 결과 확인
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def show_retrieval(query: str, top_k: int = 2):
    results = retriever.search(query, top_k=top_k)
    print(f"쿼리: '{query}'")
    print(f"{'─'*60}")

    fig, axes = plt.subplots(1, top_k, figsize=(6 * top_k, 5))
    if top_k == 1: axes = [axes]

    for i, r in enumerate(results):
        print(f"#{i+1} [{r.chunk_id}] hybrid={r.hybrid_score:.3f} "
              f"(dense={r.dense_score:.3f}, bm25={r.bm25_score:.3f})")
        print(f"    {r.text[:100].replace(chr(10),' ')}...")
        if r.image_path and Path(r.image_path).exists():
            img = mpimg.imread(r.image_path)
            axes[i].imshow(img)
            axes[i].set_title(f"#{i+1} p{r.page_num} ({r.content_type})\nscore={r.hybrid_score:.3f}")
            axes[i].axis('off')

    plt.suptitle(f"검색 결과: '{query}'", fontsize=12)
    plt.tight_layout()
    plt.show()
    print()

show_retrieval("배터리 온도 과열 DTC 코드 P0A1E")
show_retrieval("고전압 작업 안전 절차")

## 2. Qwen2.5-VL 답변 생성기 (Ollama)
검색된 페이지 이미지를 Base64로 인코딩하여 Qwen2.5-VL에 텍스트와 함께 전달합니다.
Ollama 미설치 시 자동으로 Rule-based 폴백으로 전환됩니다.

In [ ]:
# Ollama 연결 확인
def _check_ollama() -> bool:
    try:
        urllib.request.urlopen('http://localhost:11434', timeout=2)
        return True
    except:
        return False

OLLAMA_AVAILABLE = _check_ollama()
print(f"Ollama 상태: {'✓ 연결됨' if OLLAMA_AVAILABLE else '✗ 미연결 → Rule-based 폴백 사용'}")

In [ ]:
@dataclass
class RAGResponse:
    answer:      str
    sources:     list[dict]   # [{chunk_id, page_num, score}]
    model_used:  str          # 'qwen2.5-vl' | 'rule-based'
    elapsed_sec: float


class MultimodalRAG:
    """하이브리드 검색 + Qwen2.5-VL Vision LLM 답변 생성"""

    SYSTEM_PROMPT = (
        "당신은 전기차(EV) 기술 문서 전문가입니다. "
        "제공된 문서 페이지(텍스트 + 이미지)를 기반으로 질문에 정확하게 답하세요. "
        "문서에 없는 내용은 '문서에서 찾을 수 없습니다'라고 말하세요. "
        "표나 다이어그램이 있으면 그 내용도 참고하세요."
    )

    def __init__(self, retriever: HybridRetriever, top_k: int = 2,
                 ollama_model: str = 'qwen2.5vl:7b'):
        self.retriever    = retriever
        self.top_k        = top_k
        self.ollama_model = ollama_model

    def _encode_image(self, image_path: str) -> str | None:
        p = Path(image_path)
        if not p.exists():
            return None
        with open(p, 'rb') as f:
            return base64.b64encode(f.read()).decode('utf-8')

    def _build_context(self, results: list[RetrievedChunk]) -> str:
        parts = []
        for i, r in enumerate(results):
            parts.append(
                f"[문서 {i+1}: {r.doc_name} / 페이지 {r.page_num+1}]\n{r.text}"
            )
        return "\n\n---\n\n".join(parts)

    def _ollama_generate(self, query: str, results: list[RetrievedChunk]) -> str:
        import ollama
        context = self._build_context(results)
        prompt  = (
            f"다음 문서를 참고하여 질문에 답하세요.\n\n"
            f"[참고 문서]\n{context}\n\n"
            f"[질문]\n{query}\n\n"
            f"[답변]"
        )
        # 이미지 인코딩 (페이지 이미지)
        images = []
        for r in results:
            if r.image_path:
                enc = self._encode_image(r.image_path)
                if enc:
                    images.append(enc)

        msg = {'role': 'user', 'content': prompt}
        if images:
            msg['images'] = images[:2]  # 최대 2장 (컨텍스트 길이 제한)

        resp = ollama.chat(
            model=self.ollama_model,
            messages=[
                {'role': 'system', 'content': self.SYSTEM_PROMPT},
                msg
            ],
            options={'temperature': 0.1}
        )
        return resp['message']['content']

    def _rule_based_generate(self, query: str, results: list[RetrievedChunk]) -> str:
        """Ollama 미사용 시 검색 결과 요약 반환"""
        if not results:
            return "관련 문서를 찾을 수 없습니다."
        top = results[0]
        context = top.text[:600]
        return (
            f"[{top.doc_name} / 페이지 {top.page_num+1}] 관련 내용:\n\n"
            f"{context}\n\n"
            f"(※ Ollama 미연결 — 전체 AI 답변은 Ollama 실행 후 사용 가능)"
        )

    def query(self, question: str) -> RAGResponse:
        t0      = time.time()
        results = self.retriever.search(question, top_k=self.top_k)

        if OLLAMA_AVAILABLE:
            try:
                answer     = self._ollama_generate(question, results)
                model_used = self.ollama_model
            except Exception as e:
                answer     = self._rule_based_generate(question, results)
                model_used = f'rule-based (ollama error: {e})'
        else:
            answer     = self._rule_based_generate(question, results)
            model_used = 'rule-based'

        return RAGResponse(
            answer=answer,
            sources=[{'chunk_id': r.chunk_id, 'page_num': r.page_num,
                      'score': r.hybrid_score} for r in results],
            model_used=model_used,
            elapsed_sec=round(time.time() - t0, 2)
        )


rag = MultimodalRAG(retriever, top_k=2)
print(f"MultimodalRAG 초기화 완료 (top_k=2, model={rag.ollama_model})")

In [ ]:
# RAG 파이프라인 E2E 테스트
test_questions = [
    "배터리 팩의 공칭 전압과 용량은 얼마인가요?",
    "DTC 코드 P0A1E는 어떤 문제를 나타내나요?",
    "SOC는 어떤 방법으로 추정하나요?",
    "고전압 작업 시 안전 절차를 알려주세요.",
]

print("=" * 60)
print("멀티모달 RAG 파이프라인 E2E 테스트")
print("=" * 60)

responses = []
for q in test_questions:
    resp = rag.query(q)
    responses.append(resp)

    print(f"\n[Q] {q}")
    print(f"[A] {resp.answer[:300]}{'...' if len(resp.answer) > 300 else ''}")
    print(f"    출처: {[s['chunk_id'] for s in resp.sources]}")
    print(f"    모델: {resp.model_used} | 응답시간: {resp.elapsed_sec}초")

## 3. 검색 정확도 평가 — Hit Rate @ K, MRR
정답 페이지를 미리 정의하고, 검색 결과가 정답을 얼마나 찾아내는지 측정합니다.

In [ ]:
# Hit Rate @ K, MRR 평가
# ground truth: (질문, 정답 페이지 번호)
eval_set = [
    ("배터리 공칭 전압",              1),  # p1: 사양 표
    ("DTC 코드 P0A1E 온도 과열",      2),  # p2: DTC 표
    ("SOC 칼만 필터 추정 알고리즘",   1),  # p1: SOC 알고리즘
    ("셀 밸런싱 기능",                0),  # p0: 시스템 개요
    ("서비스 플러그 고전압 차단",     2),  # p2: 안전 지침
    ("NCM 811 셀 화학",               1),  # p1: 사양 표
]

def evaluate(eval_set, top_k_list=(1, 2, 3)):
    results_by_k = {k: {'hits': 0, 'rr': 0.0} for k in top_k_list}

    for query, gt_page in eval_set:
        retrieved = retriever.search(query, top_k=max(top_k_list))
        ret_pages = [r.page_num for r in retrieved]

        for k in top_k_list:
            top_pages = ret_pages[:k]
            if gt_page in top_pages:
                results_by_k[k]['hits'] += 1
                rank = top_pages.index(gt_page) + 1
                results_by_k[k]['rr'] += 1 / rank

    n = len(eval_set)
    metrics = {}
    for k in top_k_list:
        metrics[k] = {
            'hit_rate': results_by_k[k]['hits'] / n,
            'mrr':      results_by_k[k]['rr']   / n
        }
    return metrics

metrics = evaluate(eval_set)

print("=" * 45)
print("검색 정확도 평가 결과")
print("=" * 45)
print(f"{'K':>4} | {'Hit Rate':>9} | {'MRR':>9}")
print("-" * 30)
for k, m in metrics.items():
    print(f"{k:>4} | {m['hit_rate']:>8.1%} | {m['mrr']:>9.3f}")

# 시각화
import matplotlib.pyplot as plt
import numpy as np

ks        = list(metrics.keys())
hit_rates = [metrics[k]['hit_rate'] for k in ks]
mrrs      = [metrics[k]['mrr'] for k in ks]

x   = np.arange(len(ks))
w   = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, hit_rates, w, label='Hit Rate@K', color='#4C72B0')
ax.bar(x + w/2, mrrs,      w, label='MRR@K',      color='#DD8452')
ax.set_xticks(x)
ax.set_xticklabels([f'K={k}' for k in ks])
ax.set_ylim(0, 1.1)
ax.set_ylabel('점수')
ax.set_title('하이브리드 검색 정확도 (Hit Rate & MRR)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# NB03 요약
avg_elapsed = sum(r.elapsed_sec for r in responses) / len(responses)

print("=" * 60)
print("NB03 완료 — RAG 파이프라인 요약")
print("=" * 60)
print(f"  검색 방식    : 하이브리드 (BGE-M3 70% + BM25 30%)")
print(f"  생성 모델    : {responses[0].model_used}")
print(f"  평균 응답시간: {avg_elapsed:.2f}초")
print(f"  Hit Rate @2  : {metrics[2]['hit_rate']:.1%}")
print(f"  MRR @2       : {metrics[2]['mrr']:.3f}")
print()
print("[다음 단계] Notebook 04 — FastAPI 서빙 + Streamlit UI + Docker")